# MIT News: full-archive links, 2021-2025

Collects links to **all** MIT News articles (not only AI-tagged ones) for the tagged-vs-all comparison. MIT's *Load More News* button calls a clean JSON API:

```
GET https://news.mit.edu/api/recent-news/{uuid}?pg=N&_format=json
```

Each item carries `title`, `dek`, `date`, and `url`, so no HTML parsing is needed. Pages are walked from newest backwards until dates fall before 2021-01-01.

Output: `./Data/MIT_Full_Archive_Links_2021_2025.csv`


In [ ]:
import random
from datetime import datetime
from pathlib import Path
from time import sleep

import pandas as pd
import requests

BASE_URL = "https://news.mit.edu"
API_UUID = "c7115ce9-d6b8-4413-8fd2-296596a6ce72"
API_URL = f"https://news.mit.edu/api/recent-news/{API_UUID}"

START_DATE = datetime(2021, 1, 1)
MAX_PAGES = 3000
CHECKPOINT_EVERY = 20
DATA_DIR = Path("./Data")
OUTPUT_PATH = DATA_DIR / "MIT_Full_Archive_Links_2021_2025.csv"

DATA_DIR.mkdir(parents=True, exist_ok=True)

def fetch_page(pg):
    resp = requests.get(API_URL, params={"pg": pg, "_format": "json"}, timeout=15)
    resp.raise_for_status()
    return resp.json()

def parse_date(date_str):
    try:
        return datetime.strptime(date_str, "%B %d, %Y")
    except (ValueError, TypeError):
        return None

In [ ]:
# Sanity check: confirm that paging works (dates should decrease with page number).
for test_pg in [1, 2, 50, 200]:
    items = fetch_page(test_pg)
    if not items:
        print(f"pg={test_pg}: empty")
        continue
    dates = [item.get("date", "") for item in items]
    print(f"pg={test_pg}: {len(items)} items, dates {dates[-1]} ~ {dates[0]}")

In [ ]:
records = []
if OUTPUT_PATH.exists():
    records = pd.read_csv(OUTPUT_PATH).to_dict("records")
    print(f"Resumed from checkpoint with {len(records)} records.")
seen_urls = {r["Link"] for r in records}
consecutive_empty = 0

for pg in range(MAX_PAGES):
    try:
        items = fetch_page(pg)
    except requests.exceptions.RequestException as exc:
        print(f"pg={pg} failed: {exc}")
        continue

    if not items:
        consecutive_empty += 1
        if consecutive_empty >= 3:
            print("Three consecutive empty pages; stopping.")
            break
        continue
    consecutive_empty = 0

    stop_due_to_date = False
    for item in items:
        rel_url = item.get("url", "")
        if not rel_url:
            continue
        full_url = rel_url if rel_url.startswith("http") else BASE_URL + rel_url

        parsed_date = parse_date(item.get("date", ""))
        if parsed_date and parsed_date < START_DATE:
            stop_due_to_date = True
            continue

        if full_url not in seen_urls:
            records.append({"Article": item.get("title", ""), "Dek": item.get("dek", ""),
                            "Date": item.get("date", ""), "Link": full_url})
            seen_urls.add(full_url)

    if stop_due_to_date:
        print(f"pg={pg}: reached articles before {START_DATE.date()}; stopping.")
        break

    sleep(random.choice([0.3, 0.5, 0.7]))
    if (pg + 1) % CHECKPOINT_EVERY == 0:
        pd.DataFrame(records).to_csv(OUTPUT_PATH, index=False)
        print(f"Checkpoint: {len(records)} records (page {pg + 1})")

links_df = pd.DataFrame(records)
links_df.to_csv(OUTPUT_PATH, index=False)
print(f"Done: {len(links_df)} links saved to {OUTPUT_PATH}")